In [7]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!pip install -q unsloth

In [8]:
import glob, json
TRAIN = glob.glob("/kaggle/input/**/finetune_train.jsonl", recursive=True)[0]
VALID = glob.glob("/kaggle/input/**/finetune_valid.jsonl", recursive=True)[0]
EVAL  = glob.glob("/kaggle/input/**/eval_prompts.jsonl", recursive=True)[0]
print(TRAIN, VALID, EVAL)

/kaggle/input/datasets/pedrop1nho19/multistore-ft/finetune_train.jsonl /kaggle/input/datasets/pedrop1nho19/multistore-ft/finetune_valid.jsonl /kaggle/input/datasets/pedrop1nho19/multistore-eval/eval_prompts.jsonl


In [9]:
from unsloth import FastLanguageModel

MAX_LEN = 3072
model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Qwen2.5-Coder-7B-Instruct", max_seq_length=MAX_LEN, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42,
)

==((====))==  Unsloth 2026.11.6: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


In [10]:
from datasets import load_dataset

data = load_dataset("json", data_files={"train": TRAIN, "valid": VALID})
data = data.map(lambda row: {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False)})
print(data)

Generating train split: 0 examples [00:00, ? examples/s]

Generating valid split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/459 [00:00<?, ? examples/s]

Map:   0%|          | 0/51 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages', 'text'],
        num_rows: 459
    })
    valid: Dataset({
        features: ['messages', 'text'],
        num_rows: 51
    })
})


In [11]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=data["train"], eval_dataset=data["valid"],
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_LEN, packing=True,
        per_device_train_batch_size=1, gradient_accumulation_steps=8,
        num_train_epochs=2, learning_rate=2e-4, optim="adamw_8bit",
        logging_steps=5, output_dir="/kaggle/working/out", report_to="none", seed=42,
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/459 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=2):   0%|          | 0/459 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/51 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=2):   0%|          | 0/51 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


Map:   0%|          | 0/317 [00:00<?, ? examples/s]

... (2 outputs left out)

Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,0.258845
10,0.145735
15,0.055601
20,0.034117
25,0.006308
30,0.007296
35,0.000873
40,0.001466
45,0.000244
50,0.000229


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/out/checkpoint-80/tokenizer_config.json.


TrainOutput(global_step=80, training_loss=0.03197104865230358, metrics={'train_runtime': 3744.6763, 'train_samples_per_second': 0.169, 'train_steps_per_second': 0.021, 'total_flos': 6.275894637968794e+16, 'train_loss': 0.03197104865230358, 'epoch': 2.0})

In [12]:
model.save_pretrained("/kaggle/working/qwen7b-sql-lora")
tokenizer.save_pretrained("/kaggle/working/qwen7b-sql-lora")
!cd /kaggle/working && zip -qr qwen7b-sql-lora.zip qwen7b-sql-lora && ls -lh

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen7b-sql-lora/tokenizer_config.json.


total 145M
drwxr-xr-x 5 root root 4.0K Nov 18 12:22 huggingface_tokenizers_cache
drwxr-xr-x 3 root root 4.0K Nov 18 13:25 out
-rw-r--r-- 1 root root  11K Nov 18 12:09 preds_qwen_base.jsonl
-rw-r--r-- 1 root root 9.0K Nov 18 12:08 preds_qwen_ft.jsonl
drwxr-xr-x 2 root root 4.0K Nov 18 13:25 qwen7b-sql-lora
-rw-r--r-- 1 root root 145M Nov 18 13:25 qwen7b-sql-lora.zip
drwxr-xr-x 3 root root 4.0K Nov 18 12:06 unsloth_compiled_cache


In [13]:
prompts = [json.loads(l) for l in open(EVAL, encoding="utf-8")]
FastLanguageModel.for_inference(model)
with open("/kaggle/working/preds_qwen7b_ft.jsonl", "w", encoding="utf-8") as f:
    for p in prompts:
        text = tokenizer.apply_chat_template(p["messages"], tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to("cuda")
        out = model.generate(**inputs, max_new_tokens=400, do_sample=False)
        sql = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        f.write(json.dumps({"n": p["n"], "sql": sql}, ensure_ascii=False) + "\n")
        print(p["n"], sql[:80].replace("\n", " "))

Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1 SELECT b.name AS marca, COUNT(st.id) AS numero_lojas FROM brands b LEFT JOIN sto


Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


2 SELECT ROUND(SUM(s.total), 2) AS faturacao FROM sales s WHERE s.sold_at >= '2025


Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


3 SELECT st.name AS loja, COUNT(e.id) AS funcionarios FROM employees e JOIN stores


... (68 outputs left out)

Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


38 NO_SQL


Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


39 NO_SQL
40 NO_SQL
